# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:
# Task 1
# ── Data prep: fossil sources only ────────────────────────────────────────────
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

region_twh = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_twh = df_fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
print(region_twh)
print(country_twh.head())

# CVD-safe palette (Okabe-Ito) — one colour per fossil source
fossil_colours = {
    'Coal':        '#D55E00',   # vermillion
    'Oil':         '#0072B2',   # blue
    'Natural Gas': '#E69F00',   # orange
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colours,
    labels={'Source': 'Fossil source', 'TWh': 'Energy (TWh)'},
    height=700, width=1100,
    template='plotly_white'
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',          # TWh only, no percentages
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)

fig.update_layout(
    title=(f'Asia is the most fossil-dependent region — {region_twh["Asia"]:,.0f} TWh, more than twice Europe '
           f'({region_twh["Europe"]:,.0f} TWh)'),
    font=dict(family='Arial', size=12),
)

# Grey out parent nodes (Region and Country) — replace any non-palette colour
fig.data[0].marker.colors = [
    c if c in fossil_colours.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.show()


Region
Asia             35045.1
Europe           15387.0
Latin America    12376.0
North America    11131.0
Middle East       9671.0
Africa            8620.0
Oceania           8142.0
Name: TWh, dtype: float64
Country
Saudi Arabia    9671.0
Japan           9442.0
China           9421.1
South Africa    8620.0
South Korea     8252.0
Name: TWh, dtype: float64


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [6]:
# Task 2 — charts
# ── Data prep: low-carbon TWh per country ─────────────────────────────────────
low_carbon = (df.loc[df['Source_Type'] == 'Low-carbon']
                .groupby('Country')['TWh'].sum()
                .reset_index()
                .sort_values('TWh', ascending=False))
low_carbon['All'] = 'Low-carbon'          # dummy root node

low_carbon_colour = '#2E75B6'             # CVD-safe blue (low-carbon, as in the lecture)

# ── Treemap ───────────────────────────────────────────────────────────────────
fig = px.treemap(
    low_carbon,
    path=['All', 'Country'],
    values='TWh',
    color='All',
    color_discrete_map={'Low-carbon': low_carbon_colour},
    labels={'TWh': 'Energy (TWh)'},
    title='Low-carbon energy (Nuclear + Hydro) by country',
    height=600, width=1000,
    template='plotly_white'
)
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)
fig.update_layout(font=dict(family='Arial', size=12))
fig.show()

# ── Horizontal bar ────────────────────────────────────────────────────────────
leader = low_carbon.iloc[0]
second = low_carbon.iloc[1]

fig = px.bar(
    low_carbon,
    x='TWh', y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=[low_carbon_colour],
    category_orders={'Country': low_carbon['Country'].tolist()},   # largest at the top
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    height=600, width=1000,
    template='plotly_white'
)
fig.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} TWh<extra></extra>',
)
fig.update_layout(
    title=(f'{leader["Country"]} leads low-carbon energy with {leader["TWh"]:,.0f} TWh — '
           f'ahead of {second["Country"]} ({second["TWh"]:,.0f} TWh)'),
    font=dict(family='Arial', size=12),
    xaxis=dict(showgrid=False, range=[0, low_carbon['TWh'].max() * 1.15]),
    yaxis=dict(title=''),
)
fig.show()


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The horizontal bar chart. All bars start from the same baseline and are sorted, so the order (France, Norway, Canada, Brazil, …) can be read straight from top to bottom, and close values such as Norway (6,174 TWh) and Canada (5,969 TWh) are still easy to tell apart by length. In the treemap the rectangles have different widths and heights, and the layout places them by size in a grid rather than in a line, so comparing areas, especially for the smaller countries, is much harder. The treemap shows that a few countries make up most of the total, but it does not show the ranking well.


## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:
# Task 3
# ── Data prep: fossil shares per region (reuse df_fossil from Task 1) ─────────
region_mix = df_fossil.groupby(['Region', 'Source'])['TWh'].sum().reset_index()

# make sure every region has a row for every source (Middle East has no coal in the data)
region_mix = (region_mix.pivot(index='Region', columns='Source', values='TWh')
                        .fillna(0).reset_index()
                        .melt(id_vars='Region', var_name='Source', value_name='TWh'))

region_mix['Share'] = region_mix['TWh'] / region_mix.groupby('Region')['TWh'].transform('sum')

# sort regions by coal share, most coal-dependent at the top
region_order = (region_mix.loc[region_mix['Source'] == 'Coal']
                          .sort_values('Share', ascending=False)['Region'].tolist())

top_region = region_order[0]
top_coal_share = region_mix.loc[(region_mix['Region'] == top_region) & (region_mix['Source'] == 'Coal'), 'Share'].iloc[0]

# labels on coal segments only
region_mix['Label'] = np.where(region_mix['Source'] == 'Coal', region_mix['Share'].map('{:.0%}'.format), '')

# coal = same colour as Task 1, the only accent; oil and gas in greys
accent_mapping = {
    'Coal':        fossil_colours['Coal'],
    'Oil':         '#BBBBBB',
    'Natural Gas': '#DDDDDD',
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.bar(
    data_frame=region_mix,
    x='Share', y='Region',
    color='Source',
    text='Label',
    orientation='h',
    color_discrete_map=accent_mapping,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   # coal on the baseline
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_name='Region',
    hover_data={'Label': False, 'Share': ':.1%', 'TWh': ':,.0f'},
    height=550, width=1000,
    template='plotly_white',
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1,
)

fig.update_layout(
    title=f'{top_region} relies most on coal — {top_coal_share:.0%} of its fossil energy comes from coal',
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.08, title=''),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title=''),
)

# every bar sums to 100% — mark the end of the scale
fig.add_annotation(
    x=1, y=0, xref='x', yref='paper',
    xanchor='right', yanchor='top', yshift=-4,
    text='100%', showarrow=False,
    font=dict(size=11, color='grey'),
)

fig.show()


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

- **Treemap:** *How much fossil energy does each region and country use in absolute terms, and which countries make up a region's total?* The rectangle sizes show that Asia uses by far the most fossil energy and that China, Japan, India and South Korea together make up that total. The stacked bar turns everything into shares, so this information about size and countries is lost.
- **100% stacked bar:** *Which region depends most on coal compared with its oil and gas use?* Because every bar has the same length and coal starts on a common baseline, you can rank the regions by coal share directly (Africa 78%, Asia 49%, Oceania 36%, …). In the treemap the coal rectangles sit inside regions of very different sizes, so their shares cannot be compared accurately.
